# Step 3B — Pan-Human and CellTypist annotation of all QC-passed cells

Run this notebook in the dedicated **PanHumanPy TensorFlow 2.17 environment**
that also contains CellTypist.

This notebook deliberately annotates **all QC-passed cells**, not only the
current immune/endothelial gate. That maximizes sensitivity for rare T cells
that may have failed an earlier unsupervised clustering or broad signature
filter.

Two independent reference views are generated:

1. **Pan-Human Azimuth** — hierarchical broad/medium/fine labels, confidence,
   and a 128-dimensional reference embedding.
2. **CellTypist** — `Immune_All_High` and `Immune_All_Low` predictions with
   majority voting disabled. CellTypist is used as supporting immune evidence,
   not as proof that every queried cell is immune.

Only metadata and the compact Pan-Human embedding are saved. The large count
matrix is not duplicated in this environment.


In [1]:
# ---------------------------------------------------------------------
# Environment — run before importing TensorFlow or PanHumanPy
# ---------------------------------------------------------------------
import os

GPU_ID = "0"  # different physical GPU for each parallel kernel
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"

print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


CUDA_VISIBLE_DEVICES: 0


In [2]:
from __future__ import annotations

import gc
import importlib.metadata as mdlib
import json
import re
import time
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import tensorflow as tf

import panhumanpy as ph
import celltypist
from celltypist import models

SAMPLE_INFO = {
    "Screen_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "Screen"},
    "C2D15_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "C2D15"},
}

print("PanHumanPy:", mdlib.version("panhumanpy"))
print("TensorFlow:", tf.__version__)
print("CellTypist:", mdlib.version("celltypist"))
print("AnnData:", ad.__version__)
print("Visible TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))

_gpus = tf.config.list_physical_devices("GPU")
if not _gpus:
    raise RuntimeError("TensorFlow 2.17 cannot see the selected GPU.")
for _gpu in _gpus:
    tf.config.experimental.set_memory_growth(_gpu, True)


2026-07-29 11:38:59.205649: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-29 11:38:59.450333: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-29 11:38:59.529008: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-29 11:39:50.810819: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
I0000 00:00:1785339687.643507   19940 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/b

1 Physical GPUs, 1 Logical GPUs 

PanHumanPy: 1.0.0
TensorFlow: 2.17.0
CellTypist: 1.7.1
AnnData: 0.12.19
Visible TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


/home/domino/reny28/env_backs/gpu_panhuman/.venv/lib/python3.12/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [3]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057")
PIPELINE_ROOT = PROJECT_ROOT / "tmp" / "proseg_resolvi_immune_enrichment_v1"
QC_FILTERED_ROOT = PIPELINE_ROOT / "01_qc_filtered"
REFERENCE_ROOT = PIPELINE_ROOT / "03b_reference_annotations"
REFERENCE_ROOT.mkdir(parents=True, exist_ok=True)

INCLUDE_COLON = True
SECTION_NAMES = [
    sample
    for sample, meta in SAMPLE_INFO.items()
    if INCLUDE_COLON or meta["cancer_type"] != "colon_cancer"
]
# SECTION_NAMES = ["C2D15_23_25"]  # smoke test

PANHUMAN_MODEL_VERSION = "v1"
PANHUMAN_EVAL_BATCH_SIZE = 8_192
PANHUMAN_OUTPUT_MODE = "minimal"
PANHUMAN_REFINE = True
PANHUMAN_SAVE_EMBEDDING = True
PANHUMAN_MAP_CELL_ONTOLOGY = True
PANHUMAN_CONFIDENCE_THRESHOLD = 0.50

CELLTYPIST_MODELS = ["Immune_All_High.pkl", "Immune_All_Low.pkl"]
CELLTYPIST_CELL_CHUNK = 50_000
CELLTYPIST_MODE = "prob match"
CELLTYPIST_P_THRESHOLD = 0.50
CELLTYPIST_MAJORITY_VOTING = False
CELLTYPIST_TARGET_SUM = 10_000.0

USE_EXISTING_OUTPUTS = True
OVERWRITE_OUTPUTS = False
CONTINUE_ON_ERROR = True
PIPELINE_VERSION = "2026-07-29-step3b-reference-annotation-v1"

print("Samples:", SECTION_NAMES)
print("Reference annotation root:", REFERENCE_ROOT)


Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']
Reference annotation root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03b_reference_annotations


In [4]:
# ---------------------------------------------------------------------
# Paths and generic helpers
# ---------------------------------------------------------------------
def paths_for_sample(sample: str) -> dict[str, Path]:
    out = REFERENCE_ROOT / sample
    out.mkdir(parents=True, exist_ok=True)
    return {
        "input": QC_FILTERED_ROOT / sample / f"{sample}_proseg_qc_filtered.h5ad",
        "out": out,
        "panhuman": out / f"{sample}_panhuman_annotations.parquet",
        "celltypist": out / f"{sample}_celltypist_annotations.parquet",
        "merged": out / f"{sample}_reference_annotations.parquet",
        "embedding": out / f"{sample}_panhuman_embedding.npy",
        "summary": out / f"{sample}_reference_annotation_summary.json",
        "model_labels": out / f"{sample}_celltypist_model_labels.json",
    }


def write_json(payload, path: Path) -> None:
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temp.replace(path)


def get_input_metadata(path: Path):
    backed = ad.read_h5ad(path, backed="r")
    try:
        obs_names = backed.obs_names.astype(str).copy()
        var_names = backed.var_names.astype(str).copy()
        var = backed.var.copy()
        shape = backed.shape
    finally:
        backed.file.close()
    return obs_names, var_names, var, shape


def detect_feature_names_col(var_names: pd.Index, var: pd.DataFrame):
    names = var_names.astype(str)
    ensembl_fraction = float(names.str.startswith("ENSG").mean())
    if ensembl_fraction < 0.5:
        return None
    for column in (
        "gene_symbol", "gene_name", "feature_name", "name", "symbol"
    ):
        if column in var.columns:
            return column
    return None


def normalized_confidence_column(frame: pd.DataFrame) -> str | None:
    for column in (
        "final_level_confidence",
        "final_level_softmax_prob",
        "final_level_probability",
    ):
        if column in frame.columns:
            return column
    return None


def combined_label_text(frame: pd.DataFrame) -> pd.Series:
    columns = [
        column
        for column in (
            "full_hierarchical_labels",
            "azimuth_broad",
            "azimuth_medium",
            "azimuth_fine",
            "final_level_labels",
            "level_zero_labels",
        )
        if column in frame.columns
    ]
    if not columns:
        return pd.Series("", index=frame.index, dtype="string")
    text = frame[columns[0]].astype("string").fillna("")
    for column in columns[1:]:
        text = text + " | " + frame[column].astype("string").fillna("")
    return text.str.lower()


def t_label_mask(text: pd.Series) -> np.ndarray:
    pattern = re.compile(
        r"(^|[^a-z])(t[ -]?cell|t lymphocyte|cd4|cd8|treg|regulatory t|"
        r"mait|gamma[- ]?delta|gd t|t/nk)([^a-z]|$)",
        flags=re.IGNORECASE,
    )
    return text.str.contains(pattern, regex=True, na=False).to_numpy(dtype=bool)


def nk_label_mask(text: pd.Series) -> np.ndarray:
    pattern = re.compile(
        r"(^|[^a-z])(nk|natural killer|nkt)([^a-z]|$)",
        flags=re.IGNORECASE,
    )
    return text.str.contains(pattern, regex=True, na=False).to_numpy(dtype=bool)


def align_frame(frame: pd.DataFrame, obs_names: pd.Index, label: str) -> pd.DataFrame:
    frame = frame.copy()
    frame.index = frame.index.astype(str)
    if frame.index.has_duplicates:
        raise ValueError(f"{label} index contains duplicate cell IDs.")
    missing = obs_names.difference(frame.index)
    if len(missing):
        raise KeyError(
            f"{len(missing):,} cells are missing from {label}; "
            f"examples={missing[:5].tolist()}"
        )
    return frame.reindex(obs_names)


def select_embedding(result, azimuth, n_obs: int) -> np.ndarray:
    candidates = []
    if isinstance(result, np.ndarray):
        candidates.append(("return", result))
    elif isinstance(result, dict):
        candidates.extend((str(key), value) for key, value in result.items())
    embeddings_attr = getattr(azimuth, "embeddings", None)
    if isinstance(embeddings_attr, dict):
        candidates.extend(
            (f"attribute:{key}", value)
            for key, value in embeddings_attr.items()
        )

    valid = []
    for name, value in candidates:
        array = np.asarray(value)
        if array.ndim == 2 and array.shape[0] == n_obs:
            valid.append((name, array))
    if not valid:
        raise ValueError("No Pan-Human embedding with n_cells rows was found.")
    # Prefer the widest valid embedding, normally the 128-dimensional dense layer.
    name, array = max(valid, key=lambda item: item[1].shape[1])
    print("Selected Pan-Human embedding:", name, array.shape)
    return np.asarray(array, dtype=np.float32)


In [5]:
# ---------------------------------------------------------------------
# Pan-Human annotation
# ---------------------------------------------------------------------
def run_panhuman(input_path: Path, obs_names: pd.Index, feature_names_col):
    print("Running Pan-Human Azimuth ...")
    azimuth = ph.AzimuthNN(
        str(input_path),
        feature_names_col=feature_names_col,
        annotation_pipeline="supervised",
        model_version=PANHUMAN_MODEL_VERSION,
        eval_batch_size=int(PANHUMAN_EVAL_BATCH_SIZE),
        normalization_override=False,
        output_mode=PANHUMAN_OUTPUT_MODE,
        refine=PANHUMAN_REFINE,
    )

    embedding_result = None
    if PANHUMAN_SAVE_EMBEDDING:
        embedding_result = azimuth.azimuth_embed()

    if PANHUMAN_MAP_CELL_ONTOLOGY:
        for column, include_id in (
            ("azimuth_broad", True),
            ("azimuth_fine", False),
        ):
            if column in azimuth.cells_meta.columns:
                try:
                    azimuth.map_to_cell_ontology(
                        column,
                        include_cl_id=include_id,
                    )
                except Exception as exc:
                    warnings.warn(
                        f"Cell Ontology mapping failed for {column}: {exc}"
                    )

    metadata = align_frame(azimuth.cells_meta, obs_names, "Pan-Human metadata")
    confidence_col = normalized_confidence_column(metadata)
    if confidence_col is not None:
        metadata["panhuman_final_confidence"] = pd.to_numeric(
            metadata[confidence_col], errors="coerce"
        )
    else:
        metadata["panhuman_final_confidence"] = np.nan

    label_text = combined_label_text(metadata)
    metadata["panhuman_t_candidate"] = t_label_mask(label_text)
    metadata["panhuman_nk_candidate"] = nk_label_mask(label_text)
    metadata["panhuman_t_high_confidence"] = (
        metadata["panhuman_t_candidate"].to_numpy(dtype=bool)
        & (
            metadata["panhuman_final_confidence"].fillna(0).to_numpy(dtype=float)
            >= float(PANHUMAN_CONFIDENCE_THRESHOLD)
        )
    )

    embedding = None
    if PANHUMAN_SAVE_EMBEDDING:
        embedding = select_embedding(embedding_result, azimuth, len(obs_names))

    del azimuth, embedding_result
    tf.keras.backend.clear_session()
    gc.collect()
    return metadata, embedding


In [6]:
# ---------------------------------------------------------------------
# Chunked CellTypist annotation
# ---------------------------------------------------------------------
def is_t_celltypist_label(label: str) -> bool:
    text = str(label).lower()
    terms = (
        "t cell", "t cells", "treg", "regulatory t", "cd4", "cd8",
        "mait", "gamma-delta", "gamma delta", "gd t", "nkt",
    )
    return any(term in text for term in terms)


def is_nk_celltypist_label(label: str) -> bool:
    text = str(label).lower()
    return (
        "natural killer" in text
        or "nk cell" in text
        or "nk cells" in text
        or "nkt" in text
    )


def normalized_celltypist_input(raw: ad.AnnData) -> ad.AnnData:
    X = sp.csr_matrix(raw.X, dtype=np.float32)
    X.sum_duplicates()
    X.eliminate_zeros()
    result = ad.AnnData(
        X=X,
        obs=pd.DataFrame(index=raw.obs_names.copy()),
        var=pd.DataFrame(index=raw.var_names.copy()),
    )
    sc.pp.normalize_total(result, target_sum=float(CELLTYPIST_TARGET_SUM))
    sc.pp.log1p(result)
    return result


def annotate_celltypist_model(
    normalized: ad.AnnData,
    model_name: str,
    prefix: str,
):
    print("Running CellTypist model:", model_name)
    model = models.Model.load(model=model_name)
    model_labels = [str(value) for value in model.cell_types]
    t_columns = [label for label in model_labels if is_t_celltypist_label(label)]
    nk_columns = [label for label in model_labels if is_nk_celltypist_label(label)]

    frames = []
    for start in range(0, normalized.n_obs, int(CELLTYPIST_CELL_CHUNK)):
        end = min(start + int(CELLTYPIST_CELL_CHUNK), normalized.n_obs)
        chunk = normalized[start:end].copy()
        predictions = celltypist.annotate(
            chunk,
            model=model,
            mode=CELLTYPIST_MODE,
            p_thres=float(CELLTYPIST_P_THRESHOLD),
            majority_voting=bool(CELLTYPIST_MAJORITY_VOTING),
        )

        predicted = predictions.predicted_labels.copy()
        predicted.index = predicted.index.astype(str)
        probability = predictions.probability_matrix.copy()
        probability.index = probability.index.astype(str)

        frame = pd.DataFrame(index=chunk.obs_names.astype(str))
        if "predicted_labels" in predicted.columns:
            frame[f"{prefix}_prob_match_label"] = predicted[
                "predicted_labels"
            ].astype(str).reindex(frame.index)
        else:
            first_column = predicted.columns[0]
            frame[f"{prefix}_prob_match_label"] = predicted[
                first_column
            ].astype(str).reindex(frame.index)

        probability = probability.reindex(frame.index)
        frame[f"{prefix}_top_label"] = probability.idxmax(axis=1).astype(str)
        frame[f"{prefix}_top_probability"] = probability.max(axis=1).astype(float)

        if t_columns:
            frame[f"{prefix}_t_max_probability"] = probability[
                [column for column in t_columns if column in probability.columns]
            ].max(axis=1).astype(float)
        else:
            frame[f"{prefix}_t_max_probability"] = 0.0

        if nk_columns:
            frame[f"{prefix}_nk_max_probability"] = probability[
                [column for column in nk_columns if column in probability.columns]
            ].max(axis=1).astype(float)
        else:
            frame[f"{prefix}_nk_max_probability"] = 0.0

        frame[f"{prefix}_t_candidate"] = (
            frame[f"{prefix}_t_max_probability"]
            >= float(CELLTYPIST_P_THRESHOLD)
        )
        frame[f"{prefix}_nk_candidate"] = (
            frame[f"{prefix}_nk_max_probability"]
            >= float(CELLTYPIST_P_THRESHOLD)
        )
        frames.append(frame)
        print(f"CellTypist {model_name}: {end:,}/{normalized.n_obs:,} cells")

        del chunk, predictions, predicted, probability, frame
        gc.collect()

    output = pd.concat(frames, axis=0).reindex(normalized.obs_names.astype(str))
    return output, {
        "model": model_name,
        "all_labels": model_labels,
        "t_labels": t_columns,
        "nk_labels": nk_columns,
    }


def run_celltypist(input_path: Path, obs_names: pd.Index):
    raw = ad.read_h5ad(input_path)
    if not raw.obs_names.astype(str).equals(obs_names):
        raise ValueError("CellTypist input obs_names differ from Pan-Human input.")
    normalized = normalized_celltypist_input(raw)
    del raw
    gc.collect()

    all_frames = []
    model_metadata = {}
    model_prefixes = {
        "Immune_All_High.pkl": "celltypist_high",
        "Immune_All_Low.pkl": "celltypist_low",
    }
    for model_name in CELLTYPIST_MODELS:
        frame, metadata = annotate_celltypist_model(
            normalized,
            model_name,
            model_prefixes[model_name],
        )
        all_frames.append(frame)
        model_metadata[model_name] = metadata

    merged = pd.concat(all_frames, axis=1).reindex(obs_names)
    del normalized, all_frames
    gc.collect()
    return merged, model_metadata


## Download CellTypist models once

This cell is safe to rerun. The models are cached by CellTypist.


In [7]:
models.download_models(model=CELLTYPIST_MODELS)
print("CellTypist model cache:", models.models_path)


📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 61
📂 Storing models in /home/domino/.celltypist/data/models
💾 Total models to download: 2
💾 Downloading model [1/2]: Immune_All_Low.pkl
💾 Downloading model [2/2]: Immune_All_High.pkl


CellTypist model cache: /home/domino/.celltypist/data/models


In [8]:
# ---------------------------------------------------------------------
# Per-sample reference annotation runner
# ---------------------------------------------------------------------
def process_reference_sample(sample: str) -> dict:
    paths = paths_for_sample(sample)
    print("\n" + "=" * 90)
    print("Reference annotation sample:", sample)
    print("Input:", paths["input"])

    if not paths["input"].exists():
        raise FileNotFoundError(paths["input"])

    if (
        USE_EXISTING_OUTPUTS
        and not OVERWRITE_OUTPUTS
        and paths["merged"].exists()
        and paths["summary"].exists()
        and (not PANHUMAN_SAVE_EMBEDDING or paths["embedding"].exists())
    ):
        summary = json.loads(paths["summary"].read_text(encoding="utf-8"))
        if summary.get("pipeline_version") == PIPELINE_VERSION:
            print("Reusing existing reference annotations.")
            return summary

    started = time.time()
    obs_names, var_names, var, shape = get_input_metadata(paths["input"])
    feature_names_col = detect_feature_names_col(var_names, var)
    print("Feature names column for Pan-Human:", feature_names_col)

    panhuman, embedding = run_panhuman(
        paths["input"], obs_names, feature_names_col
    )
    panhuman.to_parquet(paths["panhuman"])
    if embedding is not None:
        np.save(paths["embedding"], embedding)

    celltypist_frame, model_metadata = run_celltypist(
        paths["input"], obs_names
    )
    celltypist_frame.to_parquet(paths["celltypist"])
    write_json(model_metadata, paths["model_labels"])

    merged = pd.concat([panhuman, celltypist_frame], axis=1)
    merged.index.name = "cell_id"
    merged.to_parquet(paths["merged"])

    summary = {
        "pipeline_version": PIPELINE_VERSION,
        "sample": sample,
        **SAMPLE_INFO[sample],
        "input_h5ad": str(paths["input"]),
        "shape": list(map(int, shape)),
        "feature_names_col": feature_names_col,
        "panhuman_model_version": PANHUMAN_MODEL_VERSION,
        "panhuman_eval_batch_size": int(PANHUMAN_EVAL_BATCH_SIZE),
        "panhuman_embedding": str(paths["embedding"]) if embedding is not None else None,
        "n_panhuman_t_candidate": int(merged["panhuman_t_candidate"].sum()),
        "n_panhuman_t_high_confidence": int(
            merged["panhuman_t_high_confidence"].sum()
        ),
        "n_panhuman_nk_candidate": int(merged["panhuman_nk_candidate"].sum()),
        "n_celltypist_high_t_candidate": int(
            merged["celltypist_high_t_candidate"].sum()
        ),
        "n_celltypist_low_t_candidate": int(
            merged["celltypist_low_t_candidate"].sum()
        ),
        "merged_annotations": str(paths["merged"]),
        "runtime_minutes": (time.time() - started) / 60.0,
    }
    write_json(summary, paths["summary"])
    print("Saved:", paths["merged"])

    del panhuman, celltypist_frame, merged, embedding
    tf.keras.backend.clear_session()
    gc.collect()
    return summary


## Run selected samples

Pan-Human uses the selected GPU. CellTypist is primarily CPU-oriented. For
parallel execution, use disjoint sample lists and different `GPU_ID` values.


In [9]:
reference_results = {}
reference_failures = {}

for sample in SECTION_NAMES:
    try:
        reference_results[sample] = process_reference_sample(sample)
    except Exception as exc:
        reference_failures[sample] = repr(exc)
        print(f"[FAILED] {sample}: {type(exc).__name__}: {exc}")
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        tf.keras.backend.clear_session()
        gc.collect()

pd.DataFrame.from_dict(reference_results, orient="index").to_csv(
    REFERENCE_ROOT / "all_samples_reference_annotation_summary.csv"
)
write_json(
    reference_failures,
    REFERENCE_ROOT / "all_samples_reference_annotation_failures.json",
)
print("Completed:", sorted(reference_results))
print("Failures:", json.dumps(reference_failures, indent=2))



Reference annotation sample: Screen_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/01_qc_filtered/Screen_39_21/Screen_39_21_proseg_qc_filtered.h5ad
Feature names column for Pan-Human: None
Running Pan-Human Azimuth ...
Download complete.

Integer counts detected by panhumanpy.ANNotate : True
Log-normalization performed by panhumanpy.ANNotate: True
To override log-normalization, set normalization_override=False
Query object:
    Total number of cells: 22950
    Total number of features: 18003
    Overlap w. feature reference: 4615 (~91%)

Splitting query data into 3 evaluation batch(es) of up to 
8192 cells.

Interpreting label predictions for consistent granularity at broad, medium, fine level(s).

Running model:


I0000 00:00:1785339783.279433   20217 service.cc:146] XLA service 0x7f92fc004bc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1785339783.279461   20217 service.cc:154]   StreamExecutor device (0): NVIDIA A10G, Compute Capability 8.6


124/256 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

I0000 00:00:1785339788.747497   20217 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


256/256 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
206/206 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step
Extracting azimuth embeddings:
256/256 ━━━━━━━━━━━━━━━━━━━━ 1s 879us/step
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 887us/step
206/206 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Selected Pan-Human embedding: return (22950, 128)


...

Running CellTypist model: Immune_All_High.pkl


🧬 4731 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
🔬 Input data has 10243 cells and 15547 genes
🔗 Matching reference genes in the model
🧬 4731 features used for prediction


CellTypist Immune_All_High.pkl: 10,243/10,243 cells
Running CellTypist model: Immune_All_Low.pkl


⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


CellTypist Immune_All_Low.pkl: 10,243/10,243 cells
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03b_reference_annotations/C2D15_23_25/C2D15_23_25_reference_annotations.parquet
Completed: ['C2D15_16_22', 'C2D15_17_26', 'C2D15_18_23', 'C2D15_23_25', 'C2D15_30_16', 'C2D15_39_21', 'Screen_16_22', 'Screen_17_26', 'Screen_18_23', 'Screen_23_25', 'Screen_30_16', 'Screen_39_21']
Failures: {}


In [10]:
#also record environment location
import sys
print(sys.executable)

/home/domino/reny28/env_backs/gpu_panhuman/.venv/bin/python
